## Step 1: Install the required library

Make sure you have the `openai` library installed. Use the command below if not already installed:

In [ ]:
!pip install openai==1.55.3 httpx==0.27.2

In [ ]:
# Import necessary libraries
import os
import pandas as pd
import json
from openai import OpenAI

Access data from this [link](https://themlco-my.sharepoint.com/:f:/p/chiragchauhan/Eg7biBxP1hhEta6JLchrfWgBvLawQsfxSDJO6K1BQl_J9g?e=FhiWT3)

## Step 2: Set up your API key

For security, we recommend storing your OpenAI API key in Colab Secrets. Click the '🔑' icon on the left panel, add a new secret named `OPENAI_API_KEY`, and paste your key there.

In [47]:
# Import the Colab userdata library to access secrets
from google.colab import userdata

# Retrieve the API key from Colab Secrets
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

## Step 3: Prepare the dataset

Hint: Use a JSON or CSV file and convert it to JSONL

If loading a CSV dataset and convert it to JSONL format.
Complete the conversion code below:

In [25]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
## JSONL file creation

df = pd.read_csv('GenAI_Program_FAQ_dataset.csv')
with open('data.jsonl', 'w') as f:
  for _, row in df.iterrows():
    json_line = json.dumps({
        "messages" : [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": row['prompt']},
            {"role" : "assistant", "content": row['completion']}
        ]
    })
    f.write(json_line + '\n')


## Step 4: Upload the file for fine-tuning

Use the OpenAI API to upload the dataset. Replace '<JSONL_FILE>' with your dataset file name.

In [27]:
# upload file with openai client.files
from openai import OpenAI
client = OpenAI()

client.files.create(
    file = open("data.jsonl", "rb"),
    purpose = "fine-tune"
)

FileObject(id='file-5DqztYe48YNHimbbYLt94w', bytes=8409, created_at=1775687598, filename='data.jsonl', object='file', purpose='fine-tune', status='processed', status_details=None, expires_at=None)

## Step 5: Fine-tune the model

Trigger the fine-tuning job process using the uploaded file ID. Replace '<FILE_ID>' and '<MODEL_NAME>' accordingly.

In [29]:
client.fine_tuning.jobs.create(
    training_file = "file-5DqztYe48YNHimbbYLt94w",
    model="gpt-4o-mini-2024-07-18"
)

FineTuningJob(id='ftjob-RSufUi3JfjUydbcNfkHHhGB2', created_at=1775687698, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(n_epochs='auto', batch_size='auto', learning_rate_multiplier='auto'), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-RrtQKrckuuTHBWrZf9ElcAym', result_files=[], seed=1408303896, status='validating_files', trained_tokens=None, training_file='file-5DqztYe48YNHimbbYLt94w', validation_file=None, estimated_finish=None, integrations=[], user_provided_suffix=None, metadata=None, usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None, train_experiment_id=None, eval_experiment_id=None, method={'type': 'supervised', 'supervised': {'hyperparameters': {'batch_size': 'auto', 'learning_rate_multiplier': 'auto', 'n_epochs': 'auto'}}})

## Step 6: Monitor and use the fine-tuned model

Check list of fine-tuning jobs, retrieve job details.

In [30]:
# fine_tuning.jobs.list
client.fine_tuning.jobs.list()

SyncCursorPage[FineTuningJob](data=[FineTuningJob(id='ftjob-RSufUi3JfjUydbcNfkHHhGB2', created_at=1775687698, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(n_epochs=3, batch_size=1, learning_rate_multiplier=1.8), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-RrtQKrckuuTHBWrZf9ElcAym', result_files=[], seed=1408303896, status='validating_files', trained_tokens=None, training_file='file-5DqztYe48YNHimbbYLt94w', validation_file=None, estimated_finish=None, integrations=[], user_provided_suffix=None, metadata=None, usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None, train_experiment_id=None, eval_experiment_id=None, method={'type': 'supervised', 'supervised': {'hyperparameters': {'n_epochs': 3, 'batch_size': 1, 'learning_rate_multiplier': 1.8}}}), FineTuningJob(id='ftjob-4rKyP8cj5HpcexhRp4mK96q6', created_at=

In [40]:
jobs = client.fine_tuning.jobs.list()

print("Fine-Tuning Job Details:")
for job in jobs.data:
    print(f"Job ID: {job.id}")
    print(f"  Status: {job.status}")
    if job.fine_tuned_model:
        print(f"  Fine-tuned Model ID: {job.fine_tuned_model}")
    else:
        print(f"  Fine-tuned Model ID: Not yet available ({job.status})")
    print("----------------------------------------")


Fine-Tuning Job Details:
Job ID: ftjob-okxjnHjZnysFPEHRvuq2jIYx
  Status: running
  Fine-tuned Model ID: Not yet available (running)
----------------------------------------
Job ID: ftjob-RSufUi3JfjUydbcNfkHHhGB2
  Status: failed
  Fine-tuned Model ID: Not yet available (failed)
----------------------------------------
Job ID: ftjob-4rKyP8cj5HpcexhRp4mK96q6
  Status: succeeded
  Fine-tuned Model ID: ft:gpt-4o-mini-2024-07-18:personal::DSSoTt1O
----------------------------------------
Job ID: ftjob-39PZXBhkD8vRQi1Ap7rUzQ16
  Status: failed
  Fine-tuned Model ID: Not yet available (failed)
----------------------------------------
Job ID: ftjob-BZ4wrxfUuNz7060AHj3dniGZ
  Status: succeeded
  Fine-tuned Model ID: ft:gpt-4o-mini-2024-07-18:personal:gen-ai-projects-faq:DSQEPFSj
----------------------------------------
Job ID: ftjob-oOSdQgoFRz24ojcIvR1gOO76
  Status: succeeded
  Fine-tuned Model ID: ft:gpt-4o-mini-2024-07-18:personal:genai-guidedprojects-faq:DS3tPkgW
--------------------------

In [43]:
# Completion calls can be added to test the fine-tuned model.
completions = client.chat.completions.create(
    model="ft:gpt-4o-mini-2024-07-18:personal::DSSoTt1O",
    messages=[
     {"role": "system", "content" : "Your are a helpful assistant."},
     {"role": "user", "content" : "How many projects are there in the program?"}
    ]
)

# Remember to test and validate the model after fine-tuning!
print(completions.choices[0].message.content)


There are 6+ projects in the program.


## Step 7: Use the fine-tuned model

In [45]:
completion = client.chat.completions.create(
    model="ft:gpt-4o-mini-2024-07-18:personal::DSSoTt1O",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Is there a discount for early enrollment?"}
    ]
)
print("Fine-tuned model response:", completion.choices[0].message.content)


Fine-tuned model response: Yes, early bird discounts are available for those who enroll before the specified deadline.
